# Transformació de les dades de Vacarisses: Horari ---> Diari

In [29]:
import pandas as pd

df = pd.read_csv('/Users/daviddominguez/Documents/VO project Github repository/VO-evolution/Data/Vacarisses_data.csv')
df.head()

,datetime_utc,temp_mean (°C),temp_max (°C),temp_min (°C),humidity (%),precip (mm),wind_mean_10m (km/h),wind_dir_10m (degrees),wind_max_10m (km/h),pressure (hPa),radiation (W/m²),station_code
0,1996-02-16 00:00:00,-2.0,NaN,NaN,99.0,0.0,0.0,48.0,NaN,1028.2,0.0,D2
1,1996-02-16 00:30:00,-2.1,NaN,NaN,99.0,0.0,1.1,9.0,NaN,1028.2,0.0,D2
2,1996-02-16 01:00:00,-2.3,NaN,NaN,99.0,0.0,0.0,33.0,NaN,1028.3,0.0,D2
3,1996-02-16 01:30:00,-2.5,NaN,-2.6,99.0,0.0,0.0,120.0,NaN,1028.3,0.0,D2
4,1996-02-16 02:00:00,-2.2,NaN,NaN,99.0,0.0,1.4,125.0,NaN,1027.2,0.0,D2


In [30]:
df.drop(columns = ['station_code'], inplace = True)
df.head()

,datetime_utc,temp_mean (°C),temp_max (°C),temp_min (°C),humidity (%),precip (mm),wind_mean_10m (km/h),wind_dir_10m (degrees),wind_max_10m (km/h),pressure (hPa),radiation (W/m²)
0,1996-02-16 00:00:00,-2.0,NaN,NaN,99.0,0.0,0.0,48.0,NaN,1028.2,0.0
1,1996-02-16 00:30:00,-2.1,NaN,NaN,99.0,0.0,1.1,9.0,NaN,1028.2,0.0
2,1996-02-16 01:00:00,-2.3,NaN,NaN,99.0,0.0,0.0,33.0,NaN,1028.3,0.0
3,1996-02-16 01:30:00,-2.5,NaN,-2.6,99.0,0.0,0.0,120.0,NaN,1028.3,0.0
4,1996-02-16 02:00:00,-2.2,NaN,NaN,99.0,0.0,1.4,125.0,NaN,1027.2,0.0


In [ ]:
import pandas as pd
import numpy as np

# 1. Assegurar datetime
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], errors="coerce")

# 2. Funció per direcció del vent (important!)
def mean_wind_dir(deg):
    deg = deg.dropna()
    if len(deg) == 0:
        return np.nan
    rad = np.deg2rad(deg)
    sin_mean = np.mean(np.sin(rad))
    cos_mean = np.mean(np.cos(rad))
    return np.rad2deg(np.arctan2(sin_mean, cos_mean)) % 360

# 3. Agregació diària
daily_df = (
    df.groupby(df["datetime_utc"].dt.floor("D"))
      .agg(
          T_avg=("temp_mean (°C)", "mean"),
          T_max=("temp_max (°C)", "max"),
          T_min=("temp_min (°C)", "min"),
          humidity_avg=("humidity (%)", "mean"),
          precip_total=("precip (mm)", "sum"),
          wind_avg=("wind_mean_10m (km/h)", "mean"),
          wind_dir=("wind_dir_10m (degrees)", mean_wind_dir),
          wind_max=("wind_max_10m (km/h)", "max"),
          pressure_avg=("pressure (hPa)", "mean"),
          radiation_avg=("radiation (W/m²)", "mean"),
      )
      .reset_index()
)

# 4. Rename columna data
daily_df = daily_df.rename(columns={"datetime_utc": "date"})


,date,T_avg,T_max,T_min,humidity_avg,precip_total,wind_avg,wind_dir,wind_max,pressure_avg,radiation_avg
0,1996-02-16,3.375000,9.6,-2.6,72.104167,0.0,11.102083,348.369182,23.4,1025.577083,128.270833
1,1996-02-17,6.389583,14.4,1.7,66.208333,0.0,11.735417,348.881354,28.8,1023.750000,159.145833
2,1996-02-18,6.045833,17.6,-0.6,82.687500,0.0,5.322917,340.152907,21.2,1017.587500,161.812500
3,1996-02-19,7.591667,14.2,NaN,65.875000,0.0,8.845833,344.033984,32.8,1001.329167,138.229167
4,1996-02-20,3.006522,NaN,-2.0,66.695652,5.8,5.795652,299.332749,29.9,1005.221739,148.520833


In [34]:
daily_df.tail(20)

,date,T_avg,T_max,T_min,humidity_avg,precip_total,wind_avg,wind_dir,wind_max,pressure_avg,radiation_avg
10940,2026-03-13,10.006250,19.0,2.0,87.250000,0.0,NaN,NaN,NaN,1017.389583,NaN
10941,2026-03-14,7.533333,13.6,1.8,97.333333,13.0,NaN,NaN,NaN,1005.272917,NaN
10942,2026-03-15,7.747917,18.3,-0.1,60.687500,0.0,NaN,NaN,NaN,1009.520833,NaN
10943,2026-03-16,8.945833,19.9,0.0,64.187500,0.0,NaN,NaN,NaN,1019.858333,NaN
10944,2026-03-17,10.156250,20.6,1.7,81.208333,0.0,NaN,NaN,NaN,1016.600000,NaN
10945,2026-03-18,9.916667,18.4,2.0,81.979167,0.0,NaN,NaN,NaN,1012.320833,NaN
10946,2026-03-19,9.145833,13.3,2.8,80.812500,0.0,NaN,NaN,NaN,1017.650000,NaN
10947,2026-03-20,7.120833,17.2,-0.5,73.291667,0.0,NaN,NaN,NaN,1016.989583,NaN
10948,2026-03-21,7.295833,17.1,-1.7,70.708333,0.0,NaN,NaN,NaN,1013.660417,NaN
10949,2026-03-22,8.962500,17.5,0.2,76.812500,0.2,NaN,NaN,NaN,1011.108333,NaN


In [36]:
daily_df['radiation_avg'].isna().sum()

5000

In [45]:
daily_df.rename(columns = {'precip_total':'rain_mm', 'wind_avg':'avg_wind_kmh','wind_max':'max_wind_kmh'}, inplace = True)

In [46]:
daily_df.head()

,date,T_avg,T_max,T_min,humidity_avg,rain_mm,avg_wind_kmh,wind_dir,max_wind_kmh,pressure_avg,radiation_avg
0,1996-02-16,3.375000,9.6,-2.6,72.104167,0.0,11.102083,348.369182,23.4,1025.577083,128.270833
1,1996-02-17,6.389583,14.4,1.7,66.208333,0.0,11.735417,348.881354,28.8,1023.750000,159.145833
2,1996-02-18,6.045833,17.6,-0.6,82.687500,0.0,5.322917,340.152907,21.2,1017.587500,161.812500
3,1996-02-19,7.591667,14.2,NaN,65.875000,0.0,8.845833,344.033984,32.8,1001.329167,138.229167
4,1996-02-20,3.006522,NaN,-2.0,66.695652,5.8,5.795652,299.332749,29.9,1005.221739,148.520833


In [47]:
nan_percent = daily_df.isna().mean() * 100

print(nan_percent)

date              0.000000
T_avg             0.000000
T_max             0.346715
T_min             0.383212
humidity_avg      0.145985
rain_mm           0.000000
avg_wind_kmh     74.041971
wind_dir         73.877737
max_wind_kmh     63.202555
pressure_avg      0.082117
radiation_avg    45.620438
dtype: float64


In [48]:
daily_df.drop(columns=['radiation_avg','max_wind_kmh','wind_dir','avg_wind_kmh'], inplace=True)

In [49]:
daily_df.to_csv('/Users/daviddominguez/Documents/VO project Github repository/VO-evolution/Data/Vacarisses_daily_data.csv', index=False)